# 07 - Results Interpretation

**Purpose:** consolidate the final results around the project question:

> Can time-series history, social-network exposure, and review-language signals help forecast short-term shifts in community attention toward local Yelp businesses?

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

METRICS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_metrics.csv"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_predictions.csv"
PULSE_METRICS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_metrics.csv"
PULSE_PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_predictions.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
FEATURE_SUMMARY_PATH = PROCESSED_DIR / "forecasting_feature_summary.json"

metrics = pd.read_csv(METRICS_OUTPUT_PATH)
predictions = pd.read_csv(PREDICTIONS_OUTPUT_PATH)
pulse_metrics = pd.read_csv(PULSE_METRICS_OUTPUT_PATH)
pulse_predictions = pd.read_csv(PULSE_PREDICTIONS_OUTPUT_PATH)
with GRAPH_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    graph_summary = json.load(file)
with FEATURE_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

metrics.sort_values(["split", "WAPE", "MAE"])

,split,task,model,rows,MAE,RMSE,WAPE
0,primary_covid_test,review_count_regression,Baseline: last month,27624,1.311323,2.653812,0.688473
1,primary_covid_test,review_count_regression,Baseline: rolling 3-month avg,27624,1.342685,2.954533,0.704938
2,primary_covid_test,review_count_regression,ML: historical + SNA,27624,1.443832,2.965813,0.758043
3,primary_covid_test,review_count_regression,ML: historical,27624,1.445102,2.996562,0.758710
4,primary_covid_test,review_count_regression,ML: historical + NLP,27624,1.458281,3.009956,0.765628
5,primary_covid_test,review_count_regression,ML: all modalities,27624,1.473066,3.029931,0.773391
6,primary_covid_test,review_count_regression,ML: historical + business,27624,1.476581,3.066598,0.775237
7,primary_covid_test,review_count_regression,Baseline: seasonal naive,27624,2.877643,6.556732,1.510824
8,secondary_pre_covid_test,review_count_regression,ML: historical + business,13812,1.832381,3.111897,0.383840
9,secondary_pre_covid_test,review_count_regression,ML: all modalities,13812,1.841902,3.112665,0.385834


In [2]:
regression_summary_rows = []
for split_name, split_metrics in metrics.groupby("split"):
    ranked = split_metrics.sort_values("WAPE").reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    regression_summary_rows.append({
        "split": split_name,
        "best_model": best["model"],
        "best_WAPE": best["WAPE"],
        "historical_WAPE": hist["WAPE"],
        "business_WAPE": business["WAPE"],
        "sna_WAPE": sna["WAPE"],
        "nlp_WAPE": nlp["WAPE"],
        "all_modalities_WAPE": all_modalities["WAPE"],
        "all_vs_historical_relative_change": (all_modalities["WAPE"] - hist["WAPE"]) / hist["WAPE"],
        "all_vs_business_relative_change": (all_modalities["WAPE"] - business["WAPE"]) / business["WAPE"],
    })
regression_summary = pd.DataFrame(regression_summary_rows)
regression_summary

,split,best_model,best_WAPE,historical_WAPE,business_WAPE,sna_WAPE,nlp_WAPE,all_modalities_WAPE,all_vs_historical_relative_change,all_vs_business_relative_change
0,primary_covid_test,Baseline: last month,0.688473,0.758710,0.775237,0.758043,0.765628,0.773391,0.019350,-0.002381
1,secondary_pre_covid_test,ML: historical + business,0.383840,0.395578,0.383840,0.394801,0.395632,0.385834,-0.024631,0.005196


In [3]:
pulse_summary_rows = []
for split_name, split_metrics in pulse_metrics.groupby("split"):
    ranked = split_metrics.sort_values(["F1", "PR_AUC"], ascending=[False, False]).reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    pulse_summary_rows.append({
        "split": split_name,
        "positive_rate": all_modalities["positive_rate"],
        "best_model": best["model"],
        "best_F1": best["F1"],
        "best_PR_AUC": best["PR_AUC"],
        "historical_F1": hist["F1"],
        "business_F1": business["F1"],
        "sna_F1": sna["F1"],
        "nlp_F1": nlp["F1"],
        "all_modalities_F1": all_modalities["F1"],
        "all_modalities_PR_AUC": all_modalities["PR_AUC"],
    })
pulse_summary = pd.DataFrame(pulse_summary_rows)
pulse_summary

,split,positive_rate,best_model,best_F1,best_PR_AUC,historical_F1,business_F1,sna_F1,nlp_F1,all_modalities_F1,all_modalities_PR_AUC
0,primary_covid_test,0.081198,Baseline: rising recent activity,0.274516,0.131891,0.211144,0.242919,0.172822,0.193638,0.220968,0.175975
1,secondary_pre_covid_test,0.112149,ML: historical + business,0.288208,0.236460,0.276397,0.288208,0.268895,0.279767,0.282000,0.218896


In [4]:
print("Social graph summary")
for key, value in graph_summary.items():
    print(f"{key}: {value}")

print("\nForecasting dataset summary")
for key, value in feature_summary.items():
    print(f"{key}: {value}")

Social graph summary
active_review_threshold: 5
threshold_candidates: [2, 3, 5, 10, 20]
edge_weight_formula: 1 + log1p(shared_business_count) + category_jaccard
reviewing_users: 245421
matched_user_profiles: 245419
active_users: 26598
graph_nodes: 26598
graph_edges: 116558
mean_edge_weight: 1.833072733525831
mean_edge_shared_business_count: 2.048550936014688
mean_edge_category_jaccard: 0.2671618532172656
connected_components: 11508
largest_component_size: 14965
isolated_active_users: 11387
community_method: weighted_louvain_largest_component
communities_assigned: 63
threshold_sensitivity_output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\active_reviewer_threshold_sensitivity.csv
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\user_network_features.csv

Forecasting dataset summary
min_total_reviews: 100
min_active_months: 36
business_count: 1151
row_count: 95533
feature_month_min: 2015-01
featur

In [5]:
for _, row in regression_summary.iterrows():
    split = row["split"]
    all_vs_hist = row["all_vs_historical_relative_change"] * 100
    all_vs_business = row["all_vs_business_relative_change"] * 100
    print(f"{split} regression:")
    print(f"  Best model: {row['best_model']} with WAPE={row['best_WAPE']:.4f}")
    print(f"  All modalities WAPE: {row['all_modalities_WAPE']:.4f}")
    print(f"  All modalities vs historical: {all_vs_hist:+.2f}%")
    print(f"  All modalities vs historical+business: {all_vs_business:+.2f}%")

print()
for _, row in pulse_summary.iterrows():
    split = row["split"]
    print(f"{split} attention pulses:")
    print(f"  Positive rate: {row['positive_rate']:.3f}")
    print(f"  Best model: {row['best_model']} with F1={row['best_F1']:.4f}, PR-AUC={row['best_PR_AUC']:.4f}")
    print(f"  All modalities F1: {row['all_modalities_F1']:.4f}, PR-AUC={row['all_modalities_PR_AUC']:.4f}")

primary_covid_test regression:
  Best model: Baseline: last month with WAPE=0.6885
  All modalities WAPE: 0.7734
  All modalities vs historical: +1.94%
  All modalities vs historical+business: -0.24%
secondary_pre_covid_test regression:
  Best model: ML: historical + business with WAPE=0.3838
  All modalities WAPE: 0.3858
  All modalities vs historical: -2.46%
  All modalities vs historical+business: +0.52%

primary_covid_test attention pulses:
  Positive rate: 0.081
  Best model: Baseline: rising recent activity with F1=0.2745, PR-AUC=0.1319
  All modalities F1: 0.2210, PR-AUC=0.1760
secondary_pre_covid_test attention pulses:
  Positive rate: 0.112
  Best model: ML: historical + business with F1=0.2882, PR-AUC=0.2365
  All modalities F1: 0.2820, PR-AUC=0.2189


## Interpretation Structure

The final project is a **community attention forecasting** study.

1. **Time series:** recent review patterns are the strongest baseline.
2. **Business metadata:** static context can help, but may include snapshot bias.
3. **SNA:** social exposure is measurable, but not causal influence.
4. **NLP:** recent language gives an additional lightweight modality.
5. **Target design:** attention pulses are more aligned with community-attention shifts than raw volume alone.

The report figures follow the pipeline: EDA justifies scope, SNA explains graph design, feature engineering defines targets, and model figures compare regression and classification separately.

## Final Position

This project should be presented as an interpretable multimodal analytics study of **local community attention dynamics**.

It uses time-series history, business metadata, social-network exposure, and lightweight review-language signals to forecast review activity and attention pulses. The main contribution is not only model performance; it is the analysis of how difficult it is to prepare, combine, evaluate, and interpret diverse data modalities responsibly.

Key limitations:

- Yelp friendship links are static.
- SNA features measure exposure, not causal influence.
- NLP features are lightweight lexicon/text-length signals.
- COVID-era disruption changes predictability.
- Review activity is a proxy for Yelp attention, not revenue or true customer volume.
- Static business metadata may include end-of-dataset information.